# 03 · TensorRT engines, speed, and byte-identical output

1. export RF-DETR and RTMPose to ONNX
2. build TensorRT engines **with a pinned timing cache** (the first machine creates it; commit it or keep it in S3;
   every other machine builds with `strict_cache=True`)
3. compare TensorRT against PyTorch/onnxruntime
4. time the pipeline with each back-end
5. determinism: run twice and compare the scout section of the .dvw byte for byte; do the same on a second instance

In [ ]:
import os, sys, json, time
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))
os.chdir(REPO)
print("working in", REPO)

In [ ]:
from vb.config import load_config
from vb.detect import export_rfdetr_onnx
from vb.pose import download_rtmpose
from vb.trt import build_engine
cfg = load_config("configs/default.yaml")
Path("engines").mkdir(exist_ok=True)
det_onnx = export_rfdetr_onnx(cfg.detector.variant, "engines/onnx", cfg.detector.weights, batch_size=cfg.video.batch)
pose_onnx = download_rtmpose(cfg.pose.model, cfg.pose.onnx_path)
print(det_onnx, pose_onnx)

In [ ]:
# timing caches live next to the engines; copy them to S3 and reuse them everywhere
CACHE_S3 = None   # e.g. "s3://YOUR-BUCKET/volleyball/trt-cache"
strict = False    # set True on every machine except the one that creates the caches
build_engine(det_onnx, cfg.detector.engine_path, "engines/rfdetr.timing.cache", fp16=True, strict_cache=strict)
build_engine(pose_onnx, cfg.pose.engine_path, "engines/rtmpose.timing.cache", fp16=True, strict_cache=strict)
print(open(Path(cfg.detector.engine_path).with_suffix(".json")).read())

In [ ]:
# TensorRT vs PyTorch on real frames from your clip
import torch, numpy as np
from vb.detect import Detector
from vb.io import fetch, VideoSource, to_gpu_rgb
cfg.video.source = "s3://YOUR-BUCKET/volleyball/clip.mp4"
src = VideoSource(fetch(cfg.video.source, cfg.video.local_cache), "auto", "cuda")
frames = [f for _, f in zip(range(cfg.video.batch), src)]
rgb, _ = to_gpu_rgb(frames)
cfg.detector.backend = "torch"; d_t = Detector(cfg.detector)(rgb)
cfg.detector.backend = "tensorrt"; d_r = Detector(cfg.detector)(rgb)
for a, b in zip(d_t, d_r):
    print(len(a.player_boxes), len(b.player_boxes), "players |", len(a.ball_xy), len(b.ball_xy), "balls")

In [ ]:
from vb.pipeline import run_offline
cfg.court.seed_pixels, cfg.court.seed_court = [], []     # <- paste from notebook 01
cfg.output.render = False
rows = []
for det_b, pose_b, gpu_crops in [("torch", "onnxruntime", False), ("torch", "onnxruntime", True), ("tensorrt", "tensorrt", True)]:
    cfg.detector.backend, cfg.pose.backend, cfg.pose.crops_on_gpu = det_b, pose_b, gpu_crops
    r = run_offline(cfg, run_name=f"bench-{det_b}-{pose_b}-{'gpu' if gpu_crops else 'cpu'}crops")
    rows.append({"detector": det_b, "pose": pose_b, "crops": "gpu" if gpu_crops else "cpu",
                 "fps": r.summary()["processing_fps"], **{k: v["ms_per_frame"] for k, v in r.timings.items() if k in ("detect", "pose", "homography", "track_ball")}})
import pandas as pd; pd.DataFrame(rows)

## Determinism
Run the same clip twice here, then run this cell on a second instance with the same caches and compare the printed hash.

In [ ]:
from vb.trt import events_fingerprint
cfg.detector.backend, cfg.pose.backend = "tensorrt", "tensorrt"
cfg.video.decoder = "nvdec"; cfg.video.allow_decoder_fallback = False
h = [events_fingerprint(run_offline(cfg, run_name=f"det{i}").dvw_path) for i in range(2)]
print(h[0]); print("identical across runs:", h[0] == h[1])